# 01 - 数据预处理

## 任务
- 调用 LongCat API 生成原始 COT（正确 + 错误）
- 数字随机替换 + 代码自动算答案（零API）
- 输出 `train_cot.json`（36k SFT） + `train_preference.json`（12k DPO）

## 策略
- **原始12k数据**：必须调用 LongCat API
- **增强24k数据**：复用COT + 换数字 + 代码算答案（不做API）

## 运行环境
- CPU/GPU 均可
- API调用量：12k条（约需数小时，建议先测10条验证）

In [ ]:
# 设置路径
import os
import sys

IN_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working' if IN_KAGGLE else os.getcwd()
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

print(f"工作目录: {os.getcwd()}")

In [ ]:
# 安装依赖
import subprocess, sys

def install(pkg):
    try: __import__(pkg)
    except: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for p in ['transformers', 'peft', 'tqdm', 'requests']:
    install(p)

print('依赖就绪')

In [ ]:
# 导入API
from scheme2_data_enhancement.generate_cot_data import run_data_pipeline

print('API导入成功')

In [ ]:
# 配置路径
if IN_KAGGLE:
    DATA_DIR = '/kaggle/input/math-solver-data'
    OUT_DIR = '/kaggle/working/data'
else:
    DATA_DIR = './data'
    OUT_DIR = './data'

os.makedirs(OUT_DIR, exist_ok=True)

train_data_path = os.path.join(DATA_DIR, 'train.json')
output_cot_path = os.path.join(OUT_DIR, 'train_cot_original.json')
output_pref_path = os.path.join(OUT_DIR, 'train_preference.json')
output_final_path = os.path.join(OUT_DIR, 'train_cot.json')

print(f"输入: {train_data_path}")
print(f"原始COT: {output_cot_path}")
print(f"偏好数据: {output_pref_path}")
print(f"最终数据: {output_final_path}")

In [ ]:
# ⚠️ API调用量配置
# 推荐：先用 max_api_samples=10 测试，验证通过后再跑全量
MAX_API_SAMPLES = None  # None=全部12k条
# MAX_API_SAMPLES = 10   # 测试用10条

AUG_PER_SAMPLE = 2      # 每条原始数据生成2条增强

print(f"API调用量: {MAX_API_SAMPLES or '全量12k条'}")
print(f"增强比例: {AUG_PER_SAMPLE}x")
print(f"预计最终数据: {MAX_API_SAMPLES or 12000} + {MAX_API_SAMPLES or 12000}*{AUG_PER_SAMPLE} = {(MAX_API_SAMPLES or 12000)*(1+AUG_PER_SAMPLE)}条")

In [ ]:
# 运行数据预处理流水线
# ⚠️ 警告：全量运行需要数小时，请确保API额度充足

run_data_pipeline(
    train_data_path=train_data_path,
    output_cot_path=output_cot_path,
    output_preference_path=output_pref_path,
    output_final_path=output_final_path,
    augment_per_sample=AUG_PER_SAMPLE,
    max_api_samples=MAX_API_SAMPLES,
    resume=True,
)

In [ ]:
# 验证输出
import json

with open(output_final_path) as f:
    data = json.load(f)
print(f'最终SFT数据: {len(data)}条')

with open(output_pref_path) as f:
    pref = json.load(f)
print(f'DPO偏好数据: {len(pref)}条')

# 显示样本
print('\nSFT样本示例:')
print(json.dumps(data[0], ensure_ascii=False, indent=2)[:500])